# 07 — Análisis de Separabilidad con variables fusionadas (supervisado)

**Pregunta:** ¿qué tan separables son los agarres y cuáles se confunden entre sí?

Este notebook usa las etiquetas, así que es un análisis **supervisado**. Sirve como referencia para el
clustering: si los agarres se separan mal incluso conociendo las etiquetas, el clustering sin etiquetas
no los va a separar mejor.

- **Datos**: `data/master/intact_boxcox_fused.csv` — salida del notebook de fusión (06): una fila por
  sujeto × agarre, con los 8 canales ya fusionados en una variable por feature. Solo se carga.
- **Features**: 8 variables fusionadas (`VAR_STD, Range, Energy, Peak_to_Peak, VAR, Max, MNF, SSC`).
- **Clases**: 7 agarres, sin reposo (`stimulus = 0`).
- **Validación**: LOSO con grupos `subject + origin_db` (los IDs de sujeto se repiten entre bases NinaPro,
  por eso el grupo es la combinación).

Para usar otro dataset solo se cambian `DATA_PATH` y `FEATURE_COLS`.

| Sección | Pregunta |
|---|---|
| 1. Carga | Dataset fusionado; solo se quita el reposo |
| 2. Fisher Discriminant Ratio | ¿Qué variables discriminan más? |
| 3. LDA multiclase en LOSO | ¿Se separan los agarres en un sujeto que el modelo no ha visto? |
| 4. Solapamiento por pares | ¿Qué pares de agarres se confunden? |
| 5. Silhouette por clase | ¿Cada agarre forma un grupo compacto? |
| 6. Proyección LDA | Visualización descriptiva |

In [ ]:
import os
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (silhouette_score, silhouette_samples, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ---------------- Parámetros ----------------
DATA_PATH = '../data/master/intact_boxcox_fused.csv'
FEATURE_COLS = ['VAR_STD', 'Range', 'Energy', 'Peak_to_Peak', 'VAR', 'Max', 'MNF', 'SSC']
SAVE_DIR = '../figures/separabilidad_fused'

# Shrinkage de Ledoit-Wolf para LDA: estabiliza la covarianza (las variables de amplitud
# siguen muy correlacionadas entre sí aunque estén fusionadas).
#   'auto' (recomendado) | None (LDA clásica, para comparar)
SHRINKAGE = 'auto'
# --------------------------------------------

REST = 0
N_FEATURES = len(FEATURE_COLS)
STIM_COLORS = ['#E53935', '#8E24AA', '#1E88E5', '#00ACC1',
               '#43A047', '#FB8C00', '#F4511E', '#6D4C41', '#546E7A', '#C0CA33']

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Dataset: {DATA_PATH} ({N_FEATURES} variables)')
print(f'Figuras en: {SAVE_DIR}')
print(f'SHRINKAGE = {SHRINKAGE!r}')


def make_lda(shrinkage=SHRINKAGE, for_transform=False):
    # 'lsqr' para clasificar; 'eigen' cuando se necesita .transform() (proyección)
    solver = 'eigen' if for_transform else 'lsqr'
    if shrinkage is None and not for_transform:
        solver = 'svd'
    return LinearDiscriminantAnalysis(solver=solver, shrinkage=shrinkage)

## 1. Carga

El dataset fusionado ya viene transformado (Box-Cox) y normalizado, así que aquí **solo se carga** y se quita
el reposo (si quedara alguno).

Los IDs de sujeto se repiten entre bases NinaPro, por eso el grupo LOSO es el ID compuesto
`subject + origin_db`.

In [ ]:
df = pd.read_csv(DATA_PATH)
missing = [c for c in ['stimulus', 'subject', 'origin_db'] + FEATURE_COLS if c not in df.columns]
if missing:
    raise KeyError(f'Faltan columnas en el CSV: {missing}')
print(f'Shape dataset original: {df.shape}')

df = df[df['stimulus'] != REST].copy().reset_index(drop=True)
df['group'] = df['subject'].astype(str) + '_' + df['origin_db'].astype(str)
print(f'Shape sin reposo:      {df.shape}')
print(f'Clases: {sorted(df["stimulus"].unique().tolist())}')
display(df['stimulus'].value_counts().sort_index().to_frame('n_muestras'))

counts = df.groupby('group').size()
print(f'Grupos LOSO (subject_origin_db): {counts.size}')
print(f'Muestras por grupo: {counts.min()} (mín) / {counts.max()} (máx)')

In [ ]:
X = df[FEATURE_COLS].values.astype(float)
y = df['stimulus'].values
groups = df['group'].values

STIMULI = sorted(np.unique(y).tolist())
stim_color_map = dict(zip(STIMULI, STIM_COLORS))
CHANCE = 1 / len(STIMULI)
print(f'X: {X.shape} | clases: {STIMULI} | grupos LOSO: {len(np.unique(groups))} | azar = {CHANCE:.3f}')

## 2. Fisher Discriminant Ratio (FDR)

FDR por variable fusionada: variabilidad **entre agarres** / variabilidad **dentro de cada agarre**.
Valor mayor → mejor separación. Se calcula también el **FDR global** (agregado sobre todas las variables).

In [ ]:
def fisher_ratio(X_col, y):
    # FDR univariado: SS_between / SS_within
    classes = np.unique(y)
    overall_mean = X_col.mean()
    ss_between = sum((y == c).sum() * (X_col[y == c].mean() - overall_mean) ** 2 for c in classes)
    ss_within = sum(((X_col[y == c] - X_col[y == c].mean()) ** 2).sum() for c in classes)
    return ss_between / ss_within if ss_within > 0 else np.nan


def fdr_global(X, y):
    # FDR agregado sobre todas las dimensiones
    classes = np.unique(y)
    overall_mean = X.mean(axis=0)
    ss_between = np.zeros(X.shape[1])
    ss_within = np.zeros(X.shape[1])
    for c in classes:
        X_c = X[y == c]
        ss_between += len(X_c) * (X_c.mean(axis=0) - overall_mean) ** 2
        ss_within += ((X_c - X_c.mean(axis=0)) ** 2).sum(axis=0)
    return ss_between.sum() / ss_within.sum() if ss_within.sum() > 0 else np.nan


fdr_values = np.array([fisher_ratio(X[:, i], y) for i in range(X.shape[1])])
fdr_global_value = fdr_global(X, y)

fdr_df = (pd.DataFrame({'Variable': FEATURE_COLS, 'FDR': fdr_values})
          .sort_values('FDR', ascending=False).reset_index(drop=True))
fdr_df.index = fdr_df.index + 1

print(f'FDR global ({N_FEATURES} variables): {fdr_global_value:.4f}')
display(fdr_df.style.format({'FDR': '{:.4f}'}))

In [ ]:
feat_palette = dict(zip(FEATURE_COLS, sns.color_palette('tab10', N_FEATURES)))

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=300)
fig.patch.set_facecolor('#F7F9FC')
ax.set_facecolor('#FFFFFF')

bars = ax.bar(fdr_df['Variable'], fdr_df['FDR'],
              color=[feat_palette[f] for f in fdr_df['Variable']], edgecolor='white', width=0.7)
ax.bar_label(bars, fmt='%.3f', fontsize=8, fontweight='bold', padding=2)
ax.axhline(fdr_global_value, color='tomato', linestyle='--', linewidth=1.2,
           label=f'FDR global ({fdr_global_value:.3f})')

ax.set_ylabel('FDR (entre / dentro de clases)', fontweight='bold')
ax.set_title('Fisher Discriminant Ratio por variable fusionada', fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#CCCCCC')
ax.tick_params(colors='#444444', labelsize=9)
ax.legend(frameon=True, fontsize=8)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fdr_features.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 3. LDA multiclase con LOSO

En cada pliegue se entrena una LDA con 86 sujetos y se predicen los 7 agarres del sujeto excluido.
Así se mide si los agarres se separan **en una persona que el modelo no ha visto**.

- `shrinkage='auto'` (Ledoit-Wolf): las variables de amplitud siguen muy correlacionadas entre sí, y la regularización estabiliza la covarianza.
- Se reporta accuracy por sujeto (media ± desviación) frente al nivel de azar (1 / número de agarres).
- La matriz de confusión acumulada es la medida principal de **solapamiento entre agarres**.

La LDA aquí no es el clasificador final del proyecto: es un medidor lineal e interpretable de separabilidad.

In [ ]:
logo = LeaveOneGroupOut()
y_pred = np.empty_like(y)
per_group = []

for train_idx, test_idx in logo.split(X, y, groups):
    clf = make_lda()
    clf.fit(X[train_idx], y[train_idx])
    pred = clf.predict(X[test_idx])
    y_pred[test_idx] = pred
    per_group.append({'grupo': groups[test_idx][0],
                      'accuracy': accuracy_score(y[test_idx], pred)})

per_group_df = pd.DataFrame(per_group)
acc_global = accuracy_score(y, y_pred)
bacc_global = balanced_accuracy_score(y, y_pred)

print(f'LOSO ({len(per_group_df)} pliegues, {N_FEATURES} variables) — LDA shrinkage={SHRINKAGE!r}')
print(f'Accuracy global:           {acc_global:.3f}')
print(f'Balanced accuracy:         {bacc_global:.3f}')
print(f'Accuracy por sujeto:       {per_group_df["accuracy"].mean():.3f} ± '
      f'{per_group_df["accuracy"].std():.3f}')
print(f'Nivel de azar:             {CHANCE:.3f}')
print(f'Sujetos por debajo del azar: {(per_group_df["accuracy"] <= CHANCE).sum()} '
      f'de {len(per_group_df)}')

cm = confusion_matrix(y, y_pred, labels=STIMULI, normalize='true')
cm_df = pd.DataFrame(cm, index=STIMULI, columns=STIMULI)
cm_df.index.name = 'Real'
cm_df.columns.name = 'Predicho'

recall_df = pd.Series(np.diag(cm), index=STIMULI, name='Recall').to_frame()
recall_df.index.name = 'Estímulo'
display(recall_df.T.style.format('{:.3f}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300,
                         gridspec_kw={'width_ratios': [1.25, 1]})
fig.patch.set_facecolor('#F7F9FC')

ax = axes[0]
sns.heatmap(cm_df, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            linewidths=0.5, linecolor='#FFFFFF', cbar_kws={'label': 'Proporción (por fila)'}, ax=ax)
ax.set_title('Matriz de confusión LDA — LOSO (normalizada por fila)', fontweight='bold')
ax.set_xlabel('Predicho', fontweight='bold')
ax.set_ylabel('Real', fontweight='bold')

ax = axes[1]
ax.set_facecolor('#FFFFFF')
ax.hist(per_group_df['accuracy'], bins=np.linspace(0, 1, 9), color='#1E88E5',
        edgecolor='white', alpha=0.9)
ax.axvline(CHANCE, color='tomato', linestyle='--', linewidth=1.2, label=f'Azar ({CHANCE:.3f})')
ax.axvline(per_group_df['accuracy'].mean(), color='#333333', linestyle='-', linewidth=1.2,
           label=f'Media ({per_group_df["accuracy"].mean():.3f})')
ax.set_xlabel(f'Accuracy del sujeto excluido ({len(STIMULI)} agarres)', fontweight='bold')
ax.set_ylabel('N.º de sujetos', fontweight='bold')
ax.set_title('Distribución de accuracy por sujeto', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=True, fontsize=8)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/lda_loso_confusion.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Confusiones más frecuentes (fuera de la diagonal, simetrizadas)
conf_pairs = [{'agarre_a': STIMULI[a], 'agarre_b': STIMULI[b],
               'confusion': (cm[a, b] + cm[b, a]) / 2}
              for a, b in itertools.combinations(range(len(STIMULI)), 2)]
conf_pairs_df = (pd.DataFrame(conf_pairs).sort_values('confusion', ascending=False)
                 .reset_index(drop=True))
conf_pairs_df.index = conf_pairs_df.index + 1
print('Top-5 pares más confundidos en la LDA multiclase:')
display(conf_pairs_df.head(5).style.format({'confusion': '{:.3f}'}))

## 4. Matriz de solapamiento entre agarres (error LDA por pares)

Para cada par `(i, j)` se ajusta una LDA solo con esas dos clases y se mide el error con **LOSO**
(un pliegue por sujeto, 2 muestras por pliegue).

A diferencia de la matriz de confusión multiclase, esta mide cada par **aislado**, sin la competencia
de los demás agarres. La diagonal es 0. Valores altos ⇒ mayor solapamiento.

In [ ]:
def lda_pair_error(i, j, X, y, groups):
    # Tasa de error LOSO de una LDA entrenada solo con los agarres i y j
    mask = np.isin(y, [i, j])
    Xp, yp, gp = X[mask], y[mask], groups[mask]
    errors, total = 0, 0
    for train_idx, test_idx in LeaveOneGroupOut().split(Xp, yp, gp):
        clf = make_lda()
        clf.fit(Xp[train_idx], yp[train_idx])
        errors += int((clf.predict(Xp[test_idx]) != yp[test_idx]).sum())
        total += len(test_idx)
    return errors / total if total > 0 else np.nan


n = len(STIMULI)
overlap = np.zeros((n, n))
for a, b in itertools.combinations(range(n), 2):
    overlap[a, b] = overlap[b, a] = lda_pair_error(STIMULI[a], STIMULI[b], X, y, groups)

overlap_df = pd.DataFrame(overlap, index=STIMULI, columns=STIMULI)
overlap_df.index.name = 'Estímulo'
print(f'Error medio de los {n * (n - 1) // 2} pares: {overlap[np.triu_indices(n, 1)].mean():.3f} '
      f'(azar en un par = 0.5)')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
fig.patch.set_facecolor('#F7F9FC')
ax.set_facecolor('#FFFFFF')

sns.heatmap(overlap_df, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0,
            linewidths=0.5, linecolor='#FFFFFF', cbar_kws={'label': 'Tasa de error'}, ax=ax)

ax.set_title('Matriz de solapamiento entre agarres — Error LDA por pares (LOSO)',
             fontweight='bold')
ax.set_xlabel('Estímulo', fontweight='bold')
ax.set_ylabel('Estímulo', fontweight='bold')
ax.tick_params(colors='#444444', labelsize=9)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/overlap_matrix.pdf', dpi=300, bbox_inches='tight')
plt.show()

pairs = [{'agarre_a': STIMULI[a], 'agarre_b': STIMULI[b], 'error': overlap[a, b]}
         for a, b in itertools.combinations(range(n), 2)]
pairs_df = pd.DataFrame(pairs).sort_values('error', ascending=False).reset_index(drop=True)
pairs_df.index = pairs_df.index + 1
print('Top-5 pares más solapados (mayor error LOSO):')
display(pairs_df.head(5).style.format({'error': '{:.3f}'}))
print('Top-3 pares menos solapados:')
display(pairs_df.tail(3).style.format({'error': '{:.3f}'}))

## 5. Silhouette por clase

`silhouette_samples` con distancia euclídea sobre las variables fusionadas, agrupado por agarre.
Cerca de 1 → la muestra está más cerca de su agarre que de los demás; cerca de 0 o negativo → solapamiento.

**Cuidado al interpretarlo:** 6 de las 8 variables son de amplitud (VAR, Energy, Range, Peak_to_Peak, Max,
VAR_STD) y están muy correlacionadas, así que en la distancia euclídea la amplitud pesa más que MNF o SSC.
La LDA (sección 3) encuentra direcciones discriminativas que la distancia euclídea no ve; por eso los dos
resultados pueden discrepar.

In [ ]:
sil_samples = silhouette_samples(X, y)
sil_global = silhouette_score(X, y)
df['silhouette'] = sil_samples

sil_class = (df.groupby('stimulus')['silhouette']
             .agg(media='mean', desv='std', n='count').round(4))
sil_class.index.name = 'Estímulo'
print(f'Silhouette global: {sil_global:.4f}')
display(sil_class)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=300)
fig.patch.set_facecolor('#F7F9FC')
ax.set_facecolor('#FFFFFF')

vals = sil_class.loc[STIMULI, 'media']
errs = sil_class.loc[STIMULI, 'desv']
bars = ax.bar([str(s) for s in STIMULI], vals,
              color=[stim_color_map[s] for s in STIMULI], edgecolor='white', width=0.7,
              yerr=errs, capsize=4, error_kw=dict(ecolor='#555555', lw=0.8))
ax.axhline(0, color='#333333', linewidth=0.8)
ax.axhline(sil_global, color='tomato', linestyle='--', linewidth=1.2,
           label=f'Silhouette global = {sil_global:.3f}')

# Etiquetas por encima (positivos) o por debajo (negativos) de cada barra
offset = 0.01
for bar, val in zip(bars, vals):
    va = 'bottom' if val >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width() / 2, val + (offset if val >= 0 else -offset),
            f'{val:.3f}', ha='center', va=va, fontsize=8, fontweight='bold')

# Límites simétricos que incluyen valores negativos y barras de error
lim = max(np.abs(vals - errs).max(), np.abs(vals + errs).max()) * 1.15
ax.set_ylim(-lim, lim)

ax.set_xlabel('Estímulo (agarre)', fontweight='bold')
ax.set_ylabel('Silhouette medio', fontweight='bold')
ax.set_title('Silhouette por clase (sin reposo)', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#CCCCCC')
ax.tick_params(colors='#444444', labelsize=9)
ax.legend(frameon=True, fontsize=8)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/silhouette_per_class.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 6. Proyección LDA (descriptiva)

LDA ajustada con **todos** los sujetos, solo para visualizar. Como usa todas las etiquetas, la separación
que se ve aquí es optimista: las cifras de separabilidad válidas son las de LOSO (secciones 3 y 4).
Máximo **min(n_clases − 1, n_variables)** componentes discriminantes.

In [ ]:
lda_vis = make_lda(for_transform=True)
X_lda = lda_vis.fit_transform(X, y)
evr = lda_vis.explained_variance_ratio_[:X_lda.shape[1]]

var_df = pd.DataFrame({
    'Componente': [f'LD{i+1}' for i in range(len(evr))],
    'Varianza explicada': evr,
    'Acumulada': np.cumsum(evr),
})
display(var_df.style.format({'Varianza explicada': '{:.4f}', 'Acumulada': '{:.4f}'}))
print(f'LD1 + LD2 explican el {evr[:2].sum()*100:.2f}% de la varianza discriminante')

fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
fig.patch.set_facecolor('#F7F9FC')
ax.set_facecolor('#FFFFFF')

for stim, color in zip(STIMULI, STIM_COLORS):
    mask = y == stim
    ax.scatter(X_lda[mask, 0], X_lda[mask, 1], s=60, alpha=0.85,
               edgecolor='white', linewidth=0.6, color=color, label=f'Estímulo {stim}')

ax.set_xlabel(f'LD1 ({evr[0]*100:.1f}%)', fontweight='bold')
ax.set_ylabel(f'LD2 ({evr[1]*100:.1f}%)', fontweight='bold')
ax.set_title('Proyección LDA (descriptiva, todos los sujetos) — sin reposo', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#CCCCCC')
ax.tick_params(colors='#444444', labelsize=8)
ax.legend(title='Estímulo', frameon=True, loc='best', fontsize=8, title_fontsize=9)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/lda_projection.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 7. Resumen automático

Se genera a partir de los resultados, para que no quede desactualizado si cambias `SHRINKAGE`.

In [ ]:
fdr_var = fdr_df.set_index('Variable')['FDR']
sil_sorted = sil_class['media'].sort_values(ascending=False)

print(f'Configuración: {N_FEATURES} variables, SHRINKAGE={SHRINKAGE!r}, '
      f'{len(np.unique(groups))} sujetos, {len(STIMULI)} agarres\n')
print('Fisher Discriminant Ratio')
print(f'  FDR global = {fdr_global_value:.3f}')
print(f'  Variables más discriminativas: {", ".join(f"{v} ({x:.3f})" for v, x in fdr_var.head(3).items())}')
print(f'  Variables menos discriminativas: {", ".join(f"{v} ({x:.3f})" for v, x in fdr_var.tail(2).items())}\n')
print('LDA multiclase LOSO')
print(f'  Accuracy por sujeto = {per_group_df["accuracy"].mean():.3f} ± {per_group_df["accuracy"].std():.3f} '
      f'(azar = {CHANCE:.3f}); balanced accuracy = {bacc_global:.3f}')
print(f'  Mejor reconocido: {recall_df["Recall"].idxmax()} ({recall_df["Recall"].max():.3f}) | '
      f'peor: {recall_df["Recall"].idxmin()} ({recall_df["Recall"].min():.3f})')
print(f'  Pares más confundidos: ' + ', '.join(
    f'{r.agarre_a}–{r.agarre_b} ({r.confusion:.3f})' for r in conf_pairs_df.head(3).itertuples()) + '\n')
print('Solapamiento por pares (LDA binaria LOSO)')
print(f'  Error medio = {overlap[np.triu_indices(n, 1)].mean():.3f}')
print(f'  Más solapados: ' + ', '.join(
    f'{r.agarre_a}–{r.agarre_b} ({r.error:.3f})' for r in pairs_df.head(3).itertuples()))
print(f'  Menos solapados: ' + ', '.join(
    f'{r.agarre_a}–{r.agarre_b} ({r.error:.3f})' for r in pairs_df.tail(3).itertuples()) + '\n')
print('Silhouette')
print(f'  Global = {sil_global:.3f}; mejor clase {sil_sorted.index[0]} ({sil_sorted.iloc[0]:.3f}), '
      f'peor {sil_sorted.index[-1]} ({sil_sorted.iloc[-1]:.3f})')

## 8. Interpretación y limitaciones

*(Completar con los resultados de la sección 7.)*

**Cómo leer los resultados juntos**
- **LDA LOSO** es la cifra principal: si la accuracy por sujeto está claramente por encima del azar, los
  agarres tienen estructura **lineal** generalizable entre personas con solo las variables fusionadas.
- **Matriz de confusión + solapamiento por pares** identifican el núcleo de agarres confundibles. Esta
  lista se usa para interpretar el dendrograma del análisis jerárquico.
- **FDR por variable** indica qué variables fusionadas aportan más a la separación (p. ej., amplitud
  frente a MNF/SSC).
- **Silhouette** cercano a 0 con LDA buena significa que los agarres son separables en algunas direcciones,
  pero no en la distancia euclídea global: una advertencia para el clustering.
- Si ejecutas con `SHRINKAGE = None` y comparas, ves cuánto afecta la correlación entre variables a la LDA.

**Limitaciones**
- El λ de Box-Cox se ajustó con todos los sujetos, no dentro de cada pliegue LOSO: fuga leve que puede
  inflar ligeramente las métricas.
- El método de fusión de cada variable se eligió con η² usando todas las etiquetas, también fuera del LOSO:
  otra fuente leve de optimismo.
- La LDA solo capta separabilidad **lineal**; una separabilidad baja no descarta estructura no lineal.